In [ ]:
import pandas as pd
df=pd.read_csv('monthly_panel.csv')
df.head()

,Date,ICSA,CCSA,UNRATE,JTSJOL,CPIAUCSL,FEDFUNDS,PAYEMS
0,12/1/00,346000.0,2306400.0,3.9,5088.0,174.6,6.40,132716.0
1,1/1/01,340000.0,2395750.0,4.2,5234.0,175.6,5.98,132703.0
2,2/1/01,371250.0,2486500.0,4.2,5097.0,176.0,5.49,132788.0
3,3/1/01,387200.0,2585400.0,4.3,4762.0,176.1,5.31,132751.0
4,4/1/01,396750.0,2697250.0,4.4,4615.0,176.4,4.80,132457.0


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 301 entries, 0 to 300
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Date      301 non-null    object 
 1   ICSA      298 non-null    float64
 2   CCSA      298 non-null    float64
 3   UNRATE    297 non-null    float64
 4   JTSJOL    297 non-null    float64
 5   CPIAUCSL  298 non-null    float64
 6   FEDFUNDS  299 non-null    float64
 7   PAYEMS    297 non-null    float64
dtypes: float64(7), object(1)
memory usage: 18.9+ KB


In [ ]:
df["Date"]=pd.to_datetime(df["Date"])
df=df.sort_values("Date").set_index("Date")
df.head()

/tmp/ipython-input-3280367001.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Date"]=pd.to_datetime(df["Date"])


,ICSA,CCSA,UNRATE,JTSJOL,CPIAUCSL,FEDFUNDS,PAYEMS
Date,,,,,,,
2000-12-01,346000.0,2306400.0,3.9,5088.0,174.6,6.40,132716.0
2001-01-01,340000.0,2395750.0,4.2,5234.0,175.6,5.98,132703.0
2001-02-01,371250.0,2486500.0,4.2,5097.0,176.0,5.49,132788.0
2001-03-01,387200.0,2585400.0,4.3,4762.0,176.1,5.31,132751.0
2001-04-01,396750.0,2697250.0,4.4,4615.0,176.4,4.80,132457.0


In [ ]:
df.describe()

,ICSA,CCSA,UNRATE,JTSJOL,CPIAUCSL,FEDFUNDS,PAYEMS
count,2.980000e+02,2.980000e+02,297.000000,297.000000,298.000000,299.000000,297.000000
mean,3.757638e+05,3.102038e+06,5.718182,5461.855219,234.814758,1.840803,140528.824916
std,3.306273e+05,2.207814e+06,1.958662,2301.166354,39.631851,1.896972,9010.518281
min,1.977500e+05,1.366250e+06,3.400000,2232.000000,174.600000,0.050000,129705.000000
25%,2.415000e+05,1.946612e+06,4.200000,3735.000000,204.491500,0.140000,132492.000000
50%,3.265000e+05,2.628075e+06,5.100000,4657.000000,232.363500,1.210000,137845.000000
75%,3.987500e+05,3.499750e+06,6.500000,7016.000000,255.675500,3.380000,147660.000000
max,4.663250e+06,2.032640e+07,14.800000,12134.000000,324.368000,6.400000,159540.000000


The dataset contains 301 monthly observations from December 2000 to the present, covering key labor market and macroeconomic indicators.

All variables have very few missing values (2–4 each), which can be safely handled through linear interpolation.

The summary statistics reveal:

ICSA and CCSA exhibit extremely large spikes in 2020, corresponding to COVID-19 layoffs and unprecedented economic shutdowns.

UNRATE ranges from 3.4% to 14.8%, also showing a sharp peak during the COVID recession.

PAYEMS shows long-term job growth, with noticeable dips during recessions.

JTSJOL increases steadily, reflecting stronger labor demand after 2015.

CPIAUCSL demonstrates a smooth upward trend consistent with inflation.

FEDFUNDS highlights monetary policy cycles, including the zero lower bound period (2008–2015) and rate hikes in 2022.

Overall, the dataset exhibits meaningful economic patterns: recessions appear as spikes in unemployment and layoffs, while expansions show rising job openings and employment.

Outlier Analysis (Final EDA Section)

The dataset contains several extreme outliers, particularly in the initial claims (ICSA), continued claims (CCSA), unemployment rate (UNRATE), and payroll employment (PAYEMS) variables.
These outliers correspond to well‐known macroeconomic shocks:

• The 2001 Dot-Com Recession
• The 2008 Global Financial Crisis
• The 2020 COVID-19 Economic Shutdown

For example, ICSA spikes from typical levels of 200–400K to 4.6 million in April 2020, and CCSA reaches 20 million. Similarly, UNRATE jumps to 14.8% during COVID.

These observations are economically meaningful and represent real structural breaks in labor market conditions. Therefore, they were not removed from the dataset. Removing them would distort the true behavior of the labor market and weaken the model’s predictive performance during recessionary periods.

Feature Engineering

In [ ]:
#Log transforms
import numpy as np

df["log_ICSA"] = np.log(df["ICSA"])
df["log_CCSA"] = np.log(df["CCSA"])


In [ ]:
#Lag features
df["ICSA_lag1"] = df["ICSA"].shift(1)
df["CCSA_lag1"] = df["CCSA"].shift(1)
df["PAYEMS_lag1"] = df["PAYEMS"].shift(1)
df["JTSJOL_lag1"] = df["JTSJOL"].shift(1)
df["ICSA_lag3"] = df["ICSA"].shift(3)
df["JTSJOL_lag3"] = df["JTSJOL"].shift(3)


In [ ]:
#Percentage change features
df["ICSA_pct"] = df["ICSA"].pct_change()
df["PAYEMS_pct"] = df["PAYEMS"].pct_change()
df["JTSJOL_pct"] = df["JTSJOL"].pct_change()


/tmp/ipython-input-3576728944.py:2: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df["ICSA_pct"] = df["ICSA"].pct_change()
/tmp/ipython-input-3576728944.py:3: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df["PAYEMS_pct"] = df["PAYEMS"].pct_change()
/tmp/ipython-input-3576728944.py:4: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df["JTSJOL_pct"] = df["JTSJOL"].pct_change()


In [ ]:
#Rolling averages
df["ICSA_ma3"] = df["ICSA"].rolling(3).mean()
df["CCSA_ma3"] = df["CCSA"].rolling(3).mean()
df["UNRATE_ma3"] = df["UNRATE"].rolling(3).mean()


In [ ]:
# After creating features, drop rows with NaN (from lagging)
df = df.dropna()

Feature Engineering Summary

Feature engineering was applied to improve the predictive power of the unemployment model.
Log transformations were used for highly skewed variables (ICSA, CCSA) to reduce the influence of COVID-19 outliers.
Lag features were created because labor market variables affect unemployment with a delay.
Percent-change features captured the monthly momentum of layoffs and employment trends.
Rolling averages were used to smooth short-term volatility in claims and unemployment.
These engineered features significantly enhanced the model’s ability to detect recession patterns and predict changes in unemployment.

In [ ]:
df.head()

,ICSA,CCSA,UNRATE,JTSJOL,CPIAUCSL,FEDFUNDS,PAYEMS,log_ICSA,log_CCSA,ICSA_lag1,...,PAYEMS_lag1,JTSJOL_lag1,ICSA_lag3,JTSJOL_lag3,ICSA_pct,PAYEMS_pct,JTSJOL_pct,ICSA_ma3,CCSA_ma3,UNRATE_ma3
Date,,,,,,,,,,,,,,,,,,,,,
2001-03-01,387200.0,2585400.0,4.3,4762.0,176.1,5.31,132751.0,12.866697,14.765391,371250.0,...,132788.0,5097.0,346000.0,5088.0,0.042963,-0.000279,-0.065725,366150.000000,2.489217e+06,4.233333
2001-04-01,396750.0,2697250.0,4.4,4615.0,176.4,4.80,132457.0,12.891062,14.807743,387200.0,...,132751.0,4762.0,340000.0,5234.0,0.024664,-0.002215,-0.030869,385066.666667,2.589717e+06,4.300000
2001-05-01,394500.0,2819250.0,4.3,4425.0,177.3,4.21,132409.0,12.885374,14.851981,396750.0,...,132457.0,4615.0,371250.0,5097.0,-0.005671,-0.000362,-0.041170,392816.666667,2.700633e+06,4.333333
2001-06-01,397200.0,2947400.0,4.5,4361.0,177.7,3.97,132299.0,12.892195,14.896434,394500.0,...,132409.0,4425.0,387200.0,4762.0,0.006844,-0.000831,-0.014463,396150.000000,2.821300e+06,4.400000
2001-07-01,398000.0,3030750.0,4.6,4447.0,177.4,3.77,132171.0,12.894207,14.924321,397200.0,...,132299.0,4361.0,396750.0,4615.0,0.002014,-0.000968,0.019720,396566.666667,2.932467e+06,4.466667
